# 04 - Web Scraping: Finanznachrichten (Investing.com, RSS Feeds, Reddit)

**Ziel:** Finanznachrichten aus drei verschiedenen Web-Quellen laden als zweite Nachrichtenquelle neben EODHD.

**Quellen:**
1. **Investing.com** - Forex-Nachrichten (Web Scraping mit BeautifulSoup)
2. **RSS Feeds** - Reuters, MarketWatch, FXStreet (feedparser)
3. **Reddit** - r/Forex, r/investing (JSON API)

**Wichtig:** Web Scraping ist abhaengig von der Seitenstruktur und kann sich aendern.

---

## 1. Setup und Imports

In [ ]:
# Bibliotheken importieren
import requests
from bs4 import BeautifulSoup
import feedparser
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import json
from datetime import datetime, timedelta

# Darstellung konfigurieren
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# User-Agent Header (wichtig fuer Web Scraping)
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

print('Setup erfolgreich!')

---
## 2. Quelle 1: RSS Feeds (zuverlaessigste Methode)

RSS Feeds sind die zuverlaessigste Scraping-Methode, da sie strukturiert und stabil sind.

**Feeds:**
- MarketWatch: Devisen/Forex-Nachrichten
- FXStreet: Forex News
- Investing.com RSS: Forex
- Reuters: Business News

In [ ]:
# RSS Feed URLs fuer Forex/Finanznachrichten
RSS_FEEDS = {
    'MarketWatch_Forex': 'https://feeds.marketwatch.com/marketwatch/topstories/',
    'Investing_Forex': 'https://www.investing.com/rss/news_14.rss',
    'FXStreet': 'https://www.fxstreet.com/rss',
    'Reuters_Business': 'https://www.reutersagency.com/feed/',
    'CNBC_Finance': 'https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=10000664',
}

print(f'{len(RSS_FEEDS)} RSS Feeds konfiguriert:')
for name, url in RSS_FEEDS.items():
    print(f'  - {name}')

In [ ]:
def scrape_rss_feed(feed_name, feed_url):
    """
    Laedt Artikel aus einem RSS Feed.
    
    Parameter:
        feed_name (str): Name des Feeds
        feed_url (str): URL des RSS Feeds
    
    Rueckgabe:
        list: Liste von Artikeln als Dictionaries
    """
    print(f'Lade {feed_name}...')
    
    try:
        feed = feedparser.parse(feed_url)
        
        if feed.bozo and not feed.entries:
            print(f'  WARNUNG: Feed konnte nicht geladen werden.')
            return []
        
        articles = []
        for entry in feed.entries:
            article = {
                'source': feed_name,
                'title': entry.get('title', ''),
                'link': entry.get('link', ''),
                'published': entry.get('published', entry.get('updated', '')),
                'summary': entry.get('summary', entry.get('description', '')),
            }
            
            # HTML-Tags aus Summary entfernen
            if article['summary']:
                soup = BeautifulSoup(article['summary'], 'html.parser')
                article['summary'] = soup.get_text(strip=True)
            
            articles.append(article)
        
        print(f'  -> {len(articles)} Artikel geladen')
        return articles
        
    except Exception as e:
        print(f'  FEHLER: {e}')
        return []

print('Funktion definiert.')

In [ ]:
# Alle RSS Feeds laden
rss_articles = []

for feed_name, feed_url in RSS_FEEDS.items():
    articles = scrape_rss_feed(feed_name, feed_url)
    rss_articles.extend(articles)
    time.sleep(1)  # Hoefliche Pause zwischen Requests

# In DataFrame umwandeln
if rss_articles:
    df_rss = pd.DataFrame(rss_articles)
    
    # Datum parsen (verschiedene Formate moeglich)
    df_rss['date'] = pd.to_datetime(df_rss['published'], errors='coerce', utc=True)
    df_rss['date_only'] = df_rss['date'].dt.date
    
    print(f'\nGesamt: {len(df_rss)} Artikel aus RSS Feeds')
    print(f'\nArtikel pro Quelle:')
    print(df_rss['source'].value_counts())
else:
    df_rss = pd.DataFrame()
    print('Keine RSS-Artikel geladen.')

In [ ]:
# RSS Daten anschauen
if not df_rss.empty:
    print('Spalten:', list(df_rss.columns))
    print(f'Zeitraum: {df_rss["date"].min()} bis {df_rss["date"].max()}')
    print(f'\nBeispiel-Artikel:')
    for i, row in df_rss.head(5).iterrows():
        print(f'  [{row["source"]}] {row["title"][:80]}')

---
## 3. Quelle 2: Investing.com (Web Scraping)

**Hinweis:** Investing.com hat Anti-Scraping-Massnahmen. Falls der direkte Scraping-Ansatz blockiert wird, nutzen wir den RSS Feed als Fallback (bereits in Quelle 1 enthalten).

In [ ]:
def scrape_investing_com(pages=3):
    """
    Versucht Forex-News von Investing.com zu scrapen.
    Falls blockiert, wird der Fehler dokumentiert (Teil der Datenqualitaetsanalyse).
    """
    articles = []
    base_url = 'https://www.investing.com/news/forex-news'
    
    for page in range(1, pages + 1):
        url = f'{base_url}/{page}' if page > 1 else base_url
        print(f'Lade Seite {page}: {url}')
        
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            print(f'  HTTP Status: {response.status_code}')
            
            if response.status_code == 403:
                print('  -> Zugriff verweigert (Anti-Scraping). Wird dokumentiert.')
                continue
            
            if response.status_code != 200:
                print(f'  -> Fehler: HTTP {response.status_code}')
                continue
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Artikel-Elemente suchen (Struktur kann sich aendern!)
            article_elements = soup.find_all('article') or soup.find_all('div', class_=lambda x: x and 'article' in x.lower() if x else False)
            
            if not article_elements:
                # Alternativer Selektor
                article_elements = soup.find_all('a', {'data-test': 'article-title-link'})
            
            for el in article_elements:
                title_tag = el.find('a') if el.name != 'a' else el
                if title_tag:
                    title = title_tag.get_text(strip=True)
                    link = title_tag.get('href', '')
                    if link and not link.startswith('http'):
                        link = 'https://www.investing.com' + link
                    
                    # Datum suchen
                    date_tag = el.find('time') or el.find('span', class_=lambda x: x and 'date' in x.lower() if x else False)
                    date_str = date_tag.get('datetime', date_tag.get_text(strip=True)) if date_tag else ''
                    
                    if title:
                        articles.append({
                            'source': 'Investing.com',
                            'title': title,
                            'link': link,
                            'published': date_str,
                            'summary': '',
                        })
            
            print(f'  -> {len(article_elements)} Elemente gefunden')
            time.sleep(2)  # Hoefliche Pause
            
        except requests.exceptions.Timeout:
            print('  -> Timeout. Investing.com antwortet nicht.')
        except Exception as e:
            print(f'  -> Fehler: {e}')
    
    return articles

print('Funktion definiert.')

In [ ]:
# Investing.com scrapen
investing_articles = scrape_investing_com(pages=3)

if investing_articles:
    df_investing = pd.DataFrame(investing_articles)
    df_investing['date'] = pd.to_datetime(df_investing['published'], errors='coerce', utc=True)
    df_investing['date_only'] = df_investing['date'].dt.date
    print(f'\nGesamt: {len(df_investing)} Artikel von Investing.com')
else:
    df_investing = pd.DataFrame()
    print('\nKeine Artikel von Investing.com geladen.')
    print('-> Dies ist erwartbar wegen Anti-Scraping-Massnahmen.')
    print('-> Investing.com RSS Feed wurde bereits in Quelle 1 geladen.')

---
## 4. Quelle 3: Reddit (r/Forex, r/investing)

Reddit bietet eine JSON API ohne Authentifizierung. Wir nutzen die .json-Endung an Subreddit-URLs.

**Subreddits:**
- r/Forex - Forex-spezifische Diskussionen
- r/investing - Allgemeine Investment-Diskussionen
- r/economics - Wirtschaftsnachrichten

In [ ]:
def scrape_reddit(subreddit, sort='hot', limit=100):
    """
    Laedt Posts von einem Subreddit via JSON API.
    
    Parameter:
        subreddit (str): Name des Subreddits (ohne r/)
        sort (str): Sortierung: 'hot', 'new', 'top'
        limit (int): Anzahl Posts (max 100 pro Request)
    """
    print(f'Lade r/{subreddit} ({sort})...')
    
    url = f'https://www.reddit.com/r/{subreddit}/{sort}.json'
    params = {'limit': limit}
    
    # Reddit braucht einen spezifischen User-Agent
    headers = {'User-Agent': 'DataWrangling-FHNW-Projekt/1.0'}
    
    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        print(f'  HTTP Status: {response.status_code}')
        
        if response.status_code == 429:
            print('  -> Rate Limited. Bitte spaeter erneut versuchen.')
            return []
        
        if response.status_code != 200:
            print(f'  -> Fehler: HTTP {response.status_code}')
            return []
        
        data = response.json()
        posts = []
        
        for post in data.get('data', {}).get('children', []):
            p = post['data']
            posts.append({
                'source': f'Reddit_r/{subreddit}',
                'title': p.get('title', ''),
                'link': f'https://www.reddit.com{p.get("permalink", "")}',
                'published': datetime.fromtimestamp(p.get('created_utc', 0)).isoformat(),
                'summary': p.get('selftext', '')[:500],  # Ersten 500 Zeichen
                'score': p.get('score', 0),
                'num_comments': p.get('num_comments', 0),
                'upvote_ratio': p.get('upvote_ratio', 0),
            })
        
        print(f'  -> {len(posts)} Posts geladen')
        return posts
        
    except requests.exceptions.Timeout:
        print('  -> Timeout.')
        return []
    except Exception as e:
        print(f'  -> Fehler: {e}')
        return []

print('Funktion definiert.')

In [ ]:
# Reddit Subreddits scrapen
SUBREDDITS = {
    'Forex': ['hot', 'new'],
    'investing': ['hot'],
    'economics': ['hot'],
}

reddit_articles = []

for subreddit, sorts in SUBREDDITS.items():
    for sort in sorts:
        posts = scrape_reddit(subreddit, sort=sort, limit=100)
        reddit_articles.extend(posts)
        time.sleep(2)  # Reddit Rate Limiting beachten

if reddit_articles:
    df_reddit = pd.DataFrame(reddit_articles)
    df_reddit['date'] = pd.to_datetime(df_reddit['published'], errors='coerce', utc=True)
    df_reddit['date_only'] = df_reddit['date'].dt.date
    
    # Duplikate entfernen (gleicher Post in hot + new)
    before = len(df_reddit)
    df_reddit = df_reddit.drop_duplicates(subset='link')
    print(f'\nGesamt: {len(df_reddit)} unique Posts ({before - len(df_reddit)} Duplikate entfernt)')
    print(f'\nPosts pro Subreddit:')
    print(df_reddit['source'].value_counts())
else:
    df_reddit = pd.DataFrame()
    print('Keine Reddit-Posts geladen.')

In [ ]:
# Reddit Daten anschauen
if not df_reddit.empty:
    print('Beispiel-Posts:')
    for i, row in df_reddit.head(5).iterrows():
        print(f'  [{row["source"]}] Score:{row["score"]:>5} | {row["title"][:70]}')
    
    print(f'\nStatistiken:')
    print(f'  Durchschnittlicher Score: {df_reddit["score"].mean():.0f}')
    print(f'  Durchschnittliche Kommentare: {df_reddit["num_comments"].mean():.0f}')
    print(f'  Durchschnittliche Upvote-Ratio: {df_reddit["upvote_ratio"].mean():.2f}')

---
## 5. Alle Quellen zusammenfuehren

In [ ]:
# Gemeinsame Spalten definieren
common_cols = ['source', 'title', 'link', 'published', 'summary', 'date', 'date_only']

# DataFrames zusammenfuehren (nur gemeinsame Spalten)
dfs_to_merge = []

if not df_rss.empty:
    dfs_to_merge.append(df_rss[common_cols])
    print(f'RSS Feeds:      {len(df_rss)} Artikel')

if not df_investing.empty:
    dfs_to_merge.append(df_investing[common_cols])
    print(f'Investing.com:  {len(df_investing)} Artikel')

if not df_reddit.empty:
    # Reddit hat extra Spalten (score, num_comments) - nur gemeinsame nehmen
    dfs_to_merge.append(df_reddit[common_cols])
    print(f'Reddit:         {len(df_reddit)} Posts')

if dfs_to_merge:
    df_all = pd.concat(dfs_to_merge, ignore_index=True)
    print(f'\nGesamt zusammengefuehrt: {len(df_all)} Eintraege')
else:
    df_all = pd.DataFrame()
    print('Keine Daten zum Zusammenfuehren.')

---
## 6. Datenqualitaetspruefung

In [ ]:
if not df_all.empty:
    print('DATENQUALITAET - Web Scraping')
    print('=' * 60)
    print(f'\nGesamtzahl Eintraege: {len(df_all)}')
    print(f'Quellen: {df_all["source"].nunique()}')
    
    # Fehlende Werte
    print(f'\nFehlende Werte:')
    missing = df_all.isnull().sum()
    for col in df_all.columns:
        pct = missing[col] / len(df_all) * 100
        print(f'  {col}: {missing[col]} ({pct:.1f}%)')
    
    # Leere Strings zaehlen
    print(f'\nLeere Strings:')
    for col in ['title', 'summary']:
        empty = (df_all[col] == '').sum()
        print(f'  {col}: {empty} ({empty/len(df_all)*100:.1f}%)')
    
    # Duplikate (gleicher Titel)
    dupes = df_all.duplicated(subset='title').sum()
    print(f'\nDuplikate (gleicher Titel): {dupes}')
    
    # Zeitraum
    valid_dates = df_all['date'].dropna()
    if len(valid_dates) > 0:
        print(f'\nZeitraum: {valid_dates.min()} bis {valid_dates.max()}')
        print(f'Eintraege ohne gueltiges Datum: {df_all["date"].isna().sum()}')

---
## 7. Visualisierung

In [ ]:
if not df_all.empty:
    # 7.1 Artikel pro Quelle
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Balkendiagramm
    source_counts = df_all['source'].value_counts()
    source_counts.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
    axes[0].set_title('Artikel pro Quelle', fontsize=14)
    axes[0].set_xlabel('Anzahl')
    
    # Pie Chart
    source_group = df_all['source'].apply(
        lambda x: 'RSS Feeds' if x in ['MarketWatch_Forex', 'Investing_Forex', 'FXStreet', 'Reuters_Business', 'CNBC_Finance']
        else ('Reddit' if 'Reddit' in x else 'Investing.com')
    ).value_counts()
    source_group.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#3498db', '#e74c3c', '#2ecc71'])
    axes[1].set_title('Verteilung nach Quellentyp', fontsize=14)
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 7.2 Reddit-spezifische Analyse
if not df_reddit.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Score-Verteilung
    axes[0].hist(df_reddit['score'], bins=50, edgecolor='black', alpha=0.7, color='#FF4500')
    axes[0].set_title('Reddit Post Scores', fontsize=14)
    axes[0].set_xlabel('Score')
    axes[0].set_ylabel('Haeufigkeit')
    
    # Kommentare vs Score
    axes[1].scatter(df_reddit['score'], df_reddit['num_comments'], alpha=0.5, color='#FF4500')
    axes[1].set_title('Score vs. Kommentare', fontsize=14)
    axes[1].set_xlabel('Score')
    axes[1].set_ylabel('Anzahl Kommentare')
    
    plt.tight_layout()
    plt.show()

---
## 8. Rohdaten speichern

In [ ]:
# Rohdaten als CSV speichern
OUTPUT_DIR = '../data/raw/news/webscraping'
os.makedirs(OUTPUT_DIR, exist_ok=True)

today = datetime.now().strftime('%Y-%m-%d')

# RSS Feeds
if not df_rss.empty:
    path = os.path.join(OUTPUT_DIR, f'rss_feeds_{today}.csv')
    df_rss.to_csv(path, index=False)
    print(f'Gespeichert: {path} ({len(df_rss)} Artikel)')

# Investing.com
if not df_investing.empty:
    path = os.path.join(OUTPUT_DIR, f'investing_com_{today}.csv')
    df_investing.to_csv(path, index=False)
    print(f'Gespeichert: {path} ({len(df_investing)} Artikel)')

# Reddit
if not df_reddit.empty:
    path = os.path.join(OUTPUT_DIR, f'reddit_forex_{today}.csv')
    df_reddit.to_csv(path, index=False)
    print(f'Gespeichert: {path} ({len(df_reddit)} Posts)')

# Alles zusammen
if not df_all.empty:
    path = os.path.join(OUTPUT_DIR, f'all_scraped_news_{today}.csv')
    df_all.to_csv(path, index=False)
    print(f'\nGesamt gespeichert: {path} ({len(df_all)} Eintraege)')

print('\nAlle Web-Scraping-Rohdaten gespeichert!')

---
## 9. Zusammenfassung

### Erkenntnisse:
- **RSS Feeds:** (Ergebnisse eintragen)
- **Investing.com:** (Ergebnisse eintragen - wahrscheinlich blockiert)
- **Reddit:** (Ergebnisse eintragen)
- **Datenqualitaet:** (fehlende Werte, Duplikate, etc.)

### Vergleich mit EODHD News:
- EODHD: Strukturiert, mit Sentiment, historisch
- Web Scraping: Aktuell, verschiedene Perspektiven, aber weniger strukturiert

### Naechste Schritte:
1. Vergleichs-Notebook Yahoo vs. EODHD Forex
2. Datenbereinigung aller Quellen
3. Sentiment-Analyse auf gescrapte Nachrichten anwenden
4. Forex-Kurse mit Sentiment zusammenfuehren